# **Projeto Prático: Machine Learning & Inteligência de Mercado**
## **Análise Estratégica da Concentração no Comércio Global de Bens Criativos (Dataset OpenFCS)**

---

> **Componente Curricular:** Machine Learning aplicado à Administração  
> **Instituição:** Curso de Graduação em Administração  
> **Objetivo:** Aplicação prática de Ciência de Dados, Machine Learning e Inteligência Artificial Generativa para diagnosticar padrões de concentração e (re)configuração competitiva no comércio mundial de bens criativos, a partir do acervo aberto OpenFCS (UNCTAD, alinhado ao UNESCO Framework for Cultural Statistics 2025).

---

### Corpo Docente & Contato

| Atributo | Detalhes |
| :--- | :--- |
| **Professor** | **Sérgio Assunção Monteiro, D.Sc.** |
| **Conecte-se no LinkedIn** | [🌐 linkedin.com/in/sergio-assunção-monteiro](https://www.linkedin.com/in/sergio-assun%C3%A7%C3%A3o-monteiro-b781897b/) |
| **Currículo Lattes** | [🔬 lattes.cnpq.br/9489191035734025](http://lattes.cnpq.br/9489191035734025) |
| **Repositório GitHub** | [💻 github.com/sergiomonteiro76](https://github.com/sergiomonteiro76) |

---

### Sobre este Notebook
Este ambiente foi configurado para que os alunos atuem como **Analistas de Inteligência de Mercado**. Ao longo do semestre, com apoio de modelos de linguagem (IA) integrados ao ecossistema do Google Colab, vamos reconstruir — do dado bruto ao modelo preditivo — o diagnóstico de estrutura competitiva de um setor econômico real: o comércio internacional de bens criativos (patrimônio, audiovisual, design, música, software, livros e arquitetura). Cada aula entrega uma peça do pipeline (limpeza → estatística → modelagem → rede → texto → storytelling), que alimenta, ao final, um painel executivo de (re)concentração de mercado.

* **Diretriz de Execução:** Execute as células sequencialmente e utilize os enunciados propostos ao final de cada bloco para interagir com a IA na resolução dos desafios analíticos e na interpretação dos resultados sob a ótica de negócios.
* **Fonte de dados:** [OpenFCS Dataset](https://doi.org/10.5281/zenodo.21211053) — Monteiro & Dubeux (2026), CC-BY-4.0.
* **Material de apoio:** [Paper OpenFCS (HAL)](https://hal.science/hal-05712802v1) e apostila *Economia Criativa em Dados*.

## Aula 2 — Python para Dados: Fundamentos com pandas e NumPy
Nesta aula construímos a caixa de ferramentas de programação — estruturas de dados, NumPy e pandas — que vai sustentar todas as análises do semestre.

##**Recarregar o acervo**

In [1]:
import requests, zipfile, io, os
import pandas as pd

url = "https://zenodo.org/records/21211053/files/openfcs_v1.0.0.zip?download=1"
resp = requests.get(url)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall("openfcs")

endereco = 'openfcs/openfcs-1.0.0/data/derived'
edges = pd.read_csv(endereco + "/trade_edges.csv")
entities = pd.read_csv(endereco + "/entities.csv")

print(f"trade_edges.csv: {edges.shape[0]:,} linhas, {edges.shape[1]} colunas")

trade_edges.csv: 2,197,978 linhas, 11 colunas


### 2.1 Estruturas de dados em Python
Antes de manipular uma tabela inteira, vale entender os blocos que a formam: listas, tuplas, dicionários e conjuntos. Cada um resolve um problema diferente.

####variáveis e tipos, a partir de uma linha real

In [2]:
# Uma única linha do dataset, transformada em dicionário
primeira_linha = edges.iloc[0].to_dict()
primeira_linha

print(type(primeira_linha["economy"]))            # str
print(type(primeira_linha["year"]))               # int
print(type(primeira_linha["value_usd_millions"]))  # float

<class 'str'>
<class 'int'>
<class 'float'>


###listas, tuplas, dicionários, conjuntos

In [3]:
# Lista: coleção ORDENADA e MUTÁVEL
dominios_lista = edges["fcs_domain"].unique().tolist()
print(dominios_lista)

# Conjunto: coleção SEM ORDEM e SEM DUPLICATAS — ótimo para checar pertencimento
economias_unicas = set(edges["economy"].unique())
print(f"Número de economias distintas: {len(economias_unicas)}")

# Dicionário: pares chave-valor — ótimo para contar ocorrências
contagem_por_dominio = {}
for dominio in edges["fcs_domain"]:
    contagem_por_dominio[dominio] = contagem_por_dominio.get(dominio, 0) + 1
contagem_por_dominio

# Tupla: coleção ORDENADA e IMUTÁVEL — ótima para representar um "registro fixo"
periodo_coberto = (edges["year"].min(), edges["year"].max())
periodo_coberto

['C. Visual arts (crafts) / F. Design', 'D. Books and press', 'B. Performance and celebration / C. Visual arts', 'E. Audiovisual and interactive media', 'F. Design and creative services', 'A. Cultural and natural heritage']
Número de economias distintas: 204


(2002, 2024)

### 2.2 NumPy: arrays e o poder da vetorização
Por baixo de cada coluna do pandas existe, na verdade, um array do NumPy. A diferença de desempenho entre um loop Python puro e uma operação vetorizada é o motivo pelo qual ferramentas de dados profissionais raramente usam `for`.

####loop puro vs. vetorização

In [4]:
import numpy as np
import time

valores = edges["value_usd_millions"].to_numpy()

# Abordagem 1: loop Python puro
inicio = time.time()
soma_loop = 0
for v in valores:
    soma_loop += v
tempo_loop = time.time() - inicio

# Abordagem 2: operação vetorizada do NumPy
inicio = time.time()
soma_vetorizada = valores.sum()
tempo_vetorizado = time.time() - inicio

print(f"Loop puro:  {soma_loop:,.2f}  em {tempo_loop:.4f}s")
print(f"Vetorizado: {soma_vetorizada:,.2f}  em {tempo_vetorizado:.4f}s")
print(f"O NumPy foi {tempo_loop/tempo_vetorizado:.0f}x mais rápido")

Loop puro:  33,798,258.51  em 0.2952s
Vetorizado: 33,798,258.51  em 0.0010s
O NumPy foi 309x mais rápido


### 2.3 pandas: selecionando, filtrando e ordenando

In [5]:
# Selecionar colunas
edges[["economy", "partner", "year", "value_usd_millions"]].head()

# Filtragem booleana: fluxos acima de 100 milhões de dólares
grandes_fluxos = edges[edges["value_usd_millions"] > 100]
print(f"{len(grandes_fluxos):,} fluxos acima de US$ 100 milhões")

# .loc (por condição/rótulo) — ex.: fluxos do Brasil
edges.loc[
    edges["economy"].str.contains("Brazil", case=False, na=False),
    ["economy", "partner", "year", "value_usd_millions"]
].head()

# .iloc (por posição)
edges.iloc[0:5, 0:4]

# Ordenação: maiores fluxos bilaterais individuais
edges.sort_values("value_usd_millions", ascending=False) \
     .head(10)[["economy", "partner", "fcs_domain", "year", "value_usd_millions"]]

40,390 fluxos acima de US$ 100 milhões


,economy,partner,fcs_domain,year,value_usd_millions
1998619,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2022,82696.264
1896014,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2021,81079.480
2197838,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2024,71596.783
2101839,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2023,69995.871
1913364,China,G-77 (Group of 77),C. Visual arts (crafts) / F. Design,2022,63834.370
2016097,China,G-77 (Group of 77),C. Visual arts (crafts) / F. Design,2023,61655.022
2118450,China,G-77 (Group of 77),C. Visual arts (crafts) / F. Design,2024,61274.620
1175547,G-77 (Group of 77),"China, Hong Kong SAR",C. Visual arts (crafts) / F. Design,2014,57871.537
1588926,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2018,56689.367
1793397,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2020,55534.422


### 2.4 Agregações com groupby: agrupar, aplicar, combinar
Antes de somar por domínio, é preciso checar a coluna `resolution` — o acervo traz duas resoluções (sete domínios e uma mais fina para artesanato/design), e somar as duas juntas conta o mesmo valor duas vezes.

####checar resolução antes de agregar

In [6]:
print(edges["resolution"].value_counts())

resolution
craft_sub    1171578
cer7         1026400
Name: count, dtype: int64


####filtrar resolução + agregação

In [7]:
# Ajuste o valor abaixo conforme o que apareceu na célula anterior
RESOLUCAO_7_DOMINIOS = "SUBSTITUA_PELO_VALOR_EXATO"

edges7 = edges[edges["resolution"] == RESOLUCAO_7_DOMINIOS].copy()
print(f"Linhas após filtro de resolução: {len(edges7):,}")

por_dominio_ano = (
    edges7.groupby(["fcs_domain", "year"])["value_usd_millions"]
    .sum()
    .reset_index()
    .sort_values("value_usd_millions", ascending=False)
)
por_dominio_ano.head(10)

Linhas após filtro de resolução: 0


,fcs_domain,year,value_usd_millions


####múltiplas agregações

In [8]:
resumo_por_dominio = edges7.groupby("fcs_domain")["value_usd_millions"].agg(
    total="sum", media="mean", mediana="median", n_fluxos="count"
)
resumo_por_dominio.sort_values("total", ascending=False)

,total,media,mediana,n_fluxos
fcs_domain,,,,


## **Exercícios**

## 🧪 Exercícios Práticos — Aula 2

> **Como usar:** resolva cada exercício em uma célula de código abaixo do enunciado. Depois, leve o resultado para uma IA usando o *prompt sugerido* — adapte-o com os seus próprios números. Cole a resposta da IA em uma célula de texto e escreva, em 2-3 linhas, se você concorda com ela e por quê.

---

### Exercício 1 — Um dicionário, uma frase de negócio
**🎯 Objetivo:** praticar `.iloc()` e conversão para dicionário.

**📝 Tarefa:** Escolha qualquer linha do dataset com `.iloc[n]` e transforme em dicionário. Usando as chaves desse dicionário, escreva uma frase completa descrevendo o fluxo de comércio.

**🤖 Pergunte à IA:**
> "Aqui está um dicionário representando um fluxo de comércio: [cole o dicionário]. Escreva uma frase de negócio explicando esse dado como se fosse para um gestor não técnico."

---

### Exercício 2 — Loop vs. vetorização, de novo
**🎯 Objetivo:** praticar NumPy e entender o custo de desempenho de loops.

**📝 Tarefa:** Repita a comparação de desempenho, mas agora calculando a **média** em vez da soma. O ganho de velocidade muda?

**🤖 Pergunte à IA:**
> "Rodei um teste comparando um loop Python puro com uma operação vetorizada do NumPy e o resultado foi: [cole os tempos]. Por que essa diferença de desempenho importa em um sistema de produção que atualiza um dashboard todos os dias?"

---

### Exercício 3 — Filtragem com `.loc`
**🎯 Objetivo:** praticar filtragem booleana e `groupby` combinados.

**📝 Tarefa:** Escolha um país diferente do Brasil e filtre todos os seus fluxos de exportação. Em qual domínio esse país mais exporta (em valor total)?

**🤖 Pergunte à IA:**
> "Estes são os totais exportados por domínio para [país escolhido]: [cole os números]. O que essa concentração (ou dispersão) entre domínios pode sugerir sobre a estrutura produtiva desse país?"

---

### Exercício 4 — Agregação por ano
**🎯 Objetivo:** praticar `groupby` com o cuidado da coluna `resolution`.

**📝 Tarefa:** Usando o dataframe já filtrado pela resolução de sete domínios (`edges7`), calcule o valor total exportado por ano (todos os domínios juntos). Existe alguma queda expressiva entre dois anos consecutivos?

**🤖 Pergunte à IA:**
> "O valor total exportado em bens criativos caiu de [ano/valor] para [ano/valor]. Quais eventos econômicos globais poderiam explicar essa queda?"

---

### Exercício 5 — Os 10 maiores fluxos bilaterais
**🎯 Objetivo:** praticar `sort_values` e começar a pensar em concentração de mercado.

**📝 Tarefa:** Usando `sort_values`, encontre os 10 maiores fluxos bilaterais individuais (uma linha = um par país–parceiro–domínio–ano) de toda a base. Eles pertencem a poucos países ou estão espalhados?

**🤖 Pergunte à IA:**
> "Estes são os 10 maiores fluxos bilaterais individuais do dataset: [cole a tabela]. Isso já é evidência suficiente de concentração de mercado, ou é necessário calcular um índice formal? Por quê?"

---

> 💡 **Dica geral:** o Exercício 5 é um gancho para a Aula 6, onde vamos formalizar isso com o índice HHI.